# Gold - Dimensão Tempo

Tabela calendário dinâmica para análises temporais e evolutivas.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
table_name = 'dim_tempo'
output_path_data = f"{var_gold}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_gold_schema}.{table_name}'

In [ ]:
from pyspark.sql.functions import explode, sequence, to_date, lit, year, month, dayofmonth, date_format, quarter, dayofweek

# Gera dados de 2023 a 2026
df_tempo = spark.range(1).select(
    explode(sequence(to_date(lit("2023-01-01")), to_date(lit("2026-12-31")), lit("1 day"))).alias("data")
)

df_gold = (
    df_tempo
    .withColumn("sk_tempo", date_format(col("data"), "yyyyMMdd").cast("integer"))
    .select(
        col("sk_tempo").alias("sk_tempo"),
        col("data").alias("data"),
        year(col("data")).alias("ano"),
        month(col("data")).alias("mes"),
        dayofmonth(col("data")).alias("dia"),
        quarter(col("data")).alias("trimestre"),
        dayofweek(col("data")).alias("dia_semana"),
        date_format(col("data"), "MMMM").alias("nome_mes"),
        date_format(col("data"), "EEEE").alias("nome_dia_semana")
    )
)

In [ ]:
process_data(
    df_write=df_gold,
    tipo_carga='full',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['ano', 'mes'],
    chave_upsert='sk_tempo'
)